# System Entropy Production GAN (SNEEP 1D)

이 노트북은 `AMB_1D`의 궤적을 활용하여 상태 변화($\phi_t \to \phi_{t+1}$)에 대한
시스템 엔트로피 생성량($\Delta S_{sys}$) 예측 실험을 다룹니다.

Real vs Fake 생성 모델을 두는 것이 아니라, $t$ 시간의 분포를 Real(Class 1)로,
$t+\delta t$ 시간의 분포를 Fake(Class 0)로 간주하여 GAN Loss (BCE)를 최소화함으로써
두 시점 간의 밀도 비율($\log \frac{P(\phi_t)}{P(\phi_{t+1})}$)을 직접 추론합니다.


In [ ]:
### for local server ###
import sys
import os

CNEEP_V2_ROOT = os.path.abspath('/home/user1/CNEEP_v2')

if CNEEP_V2_ROOT not in sys.path:
    sys.path.append(CNEEP_V2_ROOT)


In [ ]:
### for colab ###
# from google.colab import drive
# drive.mount('/content/drive')
# 
# import sys, os
# CNEEP_V2_ROOT = 'drive/MyDrive/CNEEP_v2/'


In [ ]:
sys.path.append(CNEEP_V2_ROOT)
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'data', 'AMB'))
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'utils'))
sys.path.append(os.path.join(CNEEP_V2_ROOT, 'models'))

from argparse import Namespace
import numpy as np
import torch
from datetime import datetime
from utils.sampler import CartesianSeqSampler
from tqdm import tqdm
import matplotlib.pyplot as plt
from generate_trajectories_1d import ActiveModelB1D

opt = Namespace()
n_trajs = 1000
n_steps = 1000
burn_in = 10000
skip = 1
opt.device = 'cuda' if torch.cuda.is_available() else 'cpu'

opt.positional = False
opt.n_layer = 2
opt.n_channel = 32
opt.seq_len = 2  # sampler pairs (phi_t, phi_{t+1})
opt.train_batch_size = 4096
opt.val_ratio = 0.2
opt.input_scalar = 1
opt.lr = 1e-3
opt.wd = 1e-3

kwargs = {
    'Lx': 256,
    'dx': 1,
    'a': 0.125,
    'b': 0.125,
    'kappa': 8.0,
    'lam': 20.0,
    'D': 0.1,
    'dt': 0.01,
    'smooth': True,
    'backend': 'torch',
    'use_gpu': torch.cuda.is_available(),
    'bc': 'periodic',
    'epr_mu_active_only': False
}
print(f'Device: {opt.device}')


In [ ]:
# Generate AMB trajectory
train_seed = 42
np.random.seed(train_seed)
if kwargs['backend'] == 'torch':
    torch.manual_seed(train_seed)

model_amb = ActiveModelB1D(**kwargs)
print(f'[INFO] Generating TRAIN trajectory...')
trajectory = model_amb.generate_trajectories(
    n_trajectories=n_trajs, n_steps=n_steps, burn_in=burn_in
)

opt.M = trajectory.shape[0]
opt.L = trajectory.shape[1]
print(f'Trajectory shape: {trajectory.shape}')

train_val_split_idx = int(opt.M * (1 - opt.val_ratio))
train_video = torch.from_numpy(trajectory[:train_val_split_idx]).float().to(opt.device)
train_video = train_video.unsqueeze(2)
val_video = torch.from_numpy(trajectory[train_val_split_idx:]).float().to(opt.device)
val_video = val_video.unsqueeze(2)

mean = torch.mean(train_video)
std  = torch.std(train_video)
transform = lambda x: (x - mean) * opt.input_scalar / std
print(f'Mean: {mean:.4f}, Std: {std:.4f}')


In [ ]:
# Initialize SNEEP Discriminator
from models.SNEEP_GAN_1D import SNEEP_Discriminator

# The network takes a single state (phi_t or phi_tp1), so input channel = 1
opt_D = Namespace(**vars(opt))
opt_D.seq_len = 1

discriminator = SNEEP_Discriminator(opt_D).to(opt.device)

optimizer = torch.optim.Adam(discriminator.parameters(), lr=opt.lr, weight_decay=opt.wd)

M_train = train_video.size(0)
M_val   = val_video.size(0)

train_sampler = CartesianSeqSampler(
    M_train, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)
val_sampler = CartesianSeqSampler(
    M_val, opt.L, opt.seq_len, opt.train_batch_size, device=opt.device)

print('Discriminator Params:', sum(p.numel() for p in discriminator.parameters()))


In [ ]:
# ── Density Ratio Estimation (System EP) Training Loop ──
from livelossplot import PlotLosses

epochs = 1000
bce_loss = torch.nn.BCEWithLogitsLoss()

liveloss = PlotLosses()
smoothing = 0.5
smooth_train_loss = None
smooth_val_loss = None

train_losses = []
val_losses   = []

for epoch in tqdm(range(1, epochs + 1)):
    # ── Train ──
    discriminator.train()
    epoch_train_loss = 0.0
    n_train_batches = 0

    for batch_idx in range(50):  # batches per epoch
        # Sample via iterator (CartesianSeqSampler uses __next__)
        ens_idx, traj_idx = next(train_sampler)

        # traj_idx shape: (seq_len, batch_size)
        batch_size = ens_idx.shape[0]
        seq = train_video[ens_idx]  # (B, L, 1, Lx)

        # Gather frames for each sequence position
        seq_out = torch.zeros(batch_size, opt.seq_len, 1, seq.shape[-1], device=opt.device)
        for i in range(opt.seq_len):
            seq_out[:, i, 0, :] = seq[torch.arange(batch_size, device=opt.device), traj_idx[i], 0, :]

        # Normalize
        seq_out = transform(seq_out)

        # phi_t and phi_{t+dt}  shape: (B, 1, Lx)
        phi_t   = seq_out[:, 0, ...]
        phi_tp1 = seq_out[:, 1, ...]

        optimizer.zero_grad()

        # logit for t distribution (Target = 1)
        logit_t   = discriminator(phi_t)
        loss_t    = bce_loss(logit_t, torch.ones_like(logit_t))

        # logit for t+dt distribution (Target = 0)
        logit_tp1 = discriminator(phi_tp1)
        loss_tp1  = bce_loss(logit_tp1, torch.zeros_like(logit_tp1))

        # GAN Loss = log D(x_t) + log(1 - D(x_{t+dt}))
        gan_loss = loss_t + loss_tp1
        gan_loss.backward()
        optimizer.step()

        epoch_train_loss += gan_loss.item()
        n_train_batches += 1

    avg_train = epoch_train_loss / n_train_batches
    train_losses.append(avg_train)

    # ── Validation ──
    discriminator.eval()
    epoch_val_loss = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for batch_idx in range(10):  # fewer val batches
            ens_idx, traj_idx = next(val_sampler)
            batch_size = ens_idx.shape[0]
            seq = val_video[ens_idx]
            seq_out = torch.zeros(batch_size, opt.seq_len, 1, seq.shape[-1], device=opt.device)
            for i in range(opt.seq_len):
                seq_out[:, i, 0, :] = seq[torch.arange(batch_size, device=opt.device), traj_idx[i], 0, :]
            seq_out = transform(seq_out)
            phi_t   = seq_out[:, 0, ...]
            phi_tp1 = seq_out[:, 1, ...]

            logit_t   = discriminator(phi_t)
            logit_tp1 = discriminator(phi_tp1)
            loss = bce_loss(logit_t, torch.ones_like(logit_t)) + bce_loss(logit_tp1, torch.zeros_like(logit_tp1))
            epoch_val_loss += loss.item()
            n_val_batches += 1

    avg_val = epoch_val_loss / n_val_batches
    val_losses.append(avg_val)

    if smooth_train_loss is None:
        smooth_train_loss = avg_train
        smooth_val_loss = avg_val
    else:
        smooth_train_loss = smoothing * smooth_train_loss + (1 - smoothing) * avg_train
        smooth_val_loss = smoothing * smooth_val_loss + (1 - smoothing) * avg_val

    liveloss.update({'train_loss': smooth_train_loss, 'val_loss': smooth_val_loss})
    liveloss.send()


In [ ]:
# ── Scatter Plot: Discriminator Logit vs mu_eq * dphi/dt ──

# Sample a batch for visualization
ens_idx, traj_idx = next(train_sampler)
batch_size = ens_idx.shape[0]
seq = train_video[ens_idx]

seq_out_unnorm = torch.zeros(batch_size, opt.seq_len, 1, seq.shape[-1], device=opt.device)
for i in range(opt.seq_len):
    seq_out_unnorm[:, i, 0, :] = seq[torch.arange(batch_size, device=opt.device), traj_idx[i], 0, :]

phi_t_unnorm   = seq_out_unnorm[:, 0, 0, :]
phi_tp1_unnorm = seq_out_unnorm[:, 1, 0, :]

# Physical system EP proxy: mu_eq * dphi/dt
dt = kwargs['dt']
dphi_dt = (phi_tp1_unnorm - phi_t_unnorm) / dt

# Convert to float64 to match model_amb internal kernel dtype
mu_eq = model_amb._mu_eq(phi_t_unnorm.double())
system_ep_phys = (mu_eq * dphi_dt.double()).cpu().numpy().flatten()

# Neural estimation via GAN Discriminator logit
seq_out_norm = transform(seq_out_unnorm)
phi_t_norm = seq_out_norm[:, 0, ...]
with torch.no_grad():
    logit_t = discriminator(phi_t_norm).cpu().numpy().flatten()

plt.figure(figsize=(8, 6))
plt.scatter(system_ep_phys, logit_t, alpha=0.3, s=2)
plt.axhline(0, color='black', linewidth=1)
plt.axvline(0, color='black', linewidth=1)
plt.xlabel(r'Physical System EP Proxy ($\mu_{eq} \cdot \dot{\phi}$)')
plt.ylabel(r'Density Ratio Logit ($\approx \log P(\phi_t) / P(\phi_{t+dt})$)')
plt.title('Validation: GAN Discriminator vs Equation-based Dynamics')
plt.grid(True)
plt.show()


In [ ]:
# ── Save Model ──
result_folder = os.path.join(CNEEP_V2_ROOT, 'results')
current_result_folder = os.path.join(result_folder,
    f"SYS_EP_GAN-{datetime.now().strftime('%Y-%m-%d-%H%M%S')}")
os.makedirs(current_result_folder, exist_ok=True)

checkpoint = {
    'state_dict': discriminator.state_dict(),
    'opt': opt
}
checkpoint_path = os.path.join(current_result_folder, 'model_parameter.pth.tar')
torch.save(checkpoint, checkpoint_path)
print(f'Model saved to {current_result_folder}')
